In [ ]:
#| default_exp parallel

In [ ]:
#| export
from fastcore.imports import *
from fastcore.basics import *
from fastcore.foundation import *
from fastcore.meta import *
from fastcore.xtras import *
from functools import wraps

import concurrent.futures,time,asyncio
from multiprocessing import Process,Queue,Manager,set_start_method,get_all_start_methods,get_context
from threading import Thread
try:
    if sys.platform == 'darwin' and IN_NOTEBOOK: set_start_method("fork")
except: pass

In [ ]:
from fastcore.test import *
from nbdev.showdoc import *
from fastcore.nb_imports import *

# Parallel

> Threading and multiprocessing functions

In [ ]:
#| export
def threaded(
    process=False, # Create a Process instead of a Thread?
    daemon=False   # Use daemon mode?
): # The Process or Thread created, which will have `result` attr injected in once complete
    "Run `f` in a `Thread` (or `Process` if `process=True`), and returns it"
    def _r(f):
        def g(_obj_td, *args, **kwargs):
            res = f(*args, **kwargs)
            _obj_td.result = res
        @wraps(f)
        def _f(*args, **kwargs):
            Proc = get_context('fork').Process if sys.platform == 'darwin' else Process
            res = (Thread,Proc)[process](target=g, args=args, kwargs=kwargs)
            res._args = (res,)+res._args
            if daemon: res.daemon = True
            res.start()
            return res
        return _f
    if callable(process):
        o = process
        process = False
        return _r(o)
    return _r

In [ ]:
@threaded
def _1():
    time.sleep(0.05)
    print("second")
    return 5

@threaded
def _2():
    time.sleep(0.01)
    print("first")

a = _1()
_2()
time.sleep(0.1)

first
second


After the thread is complete, the return value is stored in the `result` attr.

In [ ]:
#| eval: false
a.result

5

Pass `daemon=True` to make the thread (or process) a daemon, so it won't prevent the parent from exiting. Useful for background services like webservers, where you don't want a still-running thread to block process shutdown.

In [ ]:
@threaded(daemon=True)
def f(): time.sleep(0.01)

assert f().daemon

In [ ]:
#| export
def startthread(f):
    "Like `threaded`, but start thread immediately"
    return threaded(f)()

In [ ]:
@startthread
def _():
    time.sleep(0.05)
    print("second")

@startthread
def _():
    time.sleep(0.01)
    print("first")

time.sleep(0.1)

first


second


In [ ]:
#| export
def startproc(f):
    "Like `threaded(True)`, but start Process immediately"
    return threaded(True)(f)()

In [ ]:
@startproc
def _():
    time.sleep(0.05)
    print("second")

@startproc
def _():
    time.sleep(0.01)
    print("first")

time.sleep(0.1)

first

second

In [ ]:
#| export
def _call(lock, pause, n, g, item):
    l = False
    if pause:
        try:
            l = lock.acquire(timeout=pause*(n+2))
            time.sleep(pause)
        finally:
            if l: lock.release()
    return g(item)

In [ ]:
#| export
def parallelable(param_name, num_workers, f=None):
    f_in_main = f == None or sys.modules[f.__module__].__name__ == "__main__"
    if sys.platform == "win32" and IN_NOTEBOOK and num_workers > 0 and f_in_main:
        print("Due to IPython and Windows limitation, python multiprocessing isn't available now.")
        print(f"So `{param_name}` has to be changed to 0 to avoid getting stuck")
        return False
    return True

In [ ]:
#| export
class ThreadPoolExecutor(concurrent.futures.ThreadPoolExecutor):
    "Same as Python's ThreadPoolExecutor, except can pass `max_workers==0` for serial execution"
    def __init__(self, max_workers=defaults.cpus, on_exc=print, pause=0, **kwargs):
        if max_workers is None: max_workers=defaults.cpus
        store_attr()
        self.not_parallel = max_workers==0
        if self.not_parallel: max_workers=1
        super().__init__(max_workers, **kwargs)

    def map(self, f, items, *args, timeout=None, chunksize=1, **kwargs):
        if self.not_parallel == False: self.lock = Manager().Lock()
        g = partial(f, *args, **kwargs)
        if self.not_parallel: return map(g, items)
        _g = partial(_call, self.lock, self.pause, self.max_workers, g)
        try: return super().map(_g, items, timeout=timeout, chunksize=chunksize)
        except Exception as e: self.on_exc(e)

In [ ]:
show_doc(ThreadPoolExecutor, title_level=4)

---

[source](https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/parallel.py#L76){target="_blank" style="float:right; font-size:smaller"}

#### ThreadPoolExecutor

```python

def ThreadPoolExecutor(
    max_workers:int=16, on_exc:builtin_function_or_method=print, pause:int=0, kwargs:VAR_KEYWORD
):


```

*Same as Python's ThreadPoolExecutor, except can pass `max_workers==0` for serial execution*

In [ ]:
#| export
@delegates()
class ProcessPoolExecutor(concurrent.futures.ProcessPoolExecutor):
    "Same as Python's ProcessPoolExecutor, except can pass `max_workers==0` for serial execution"
    def __init__(self, max_workers=defaults.cpus, on_exc=print, pause=0, **kwargs):
        if max_workers is None: max_workers=defaults.cpus
        store_attr()
        self.not_parallel = max_workers==0
        if self.not_parallel: max_workers=1
        super().__init__(max_workers, **kwargs)

    def map(self, f, items, *args, timeout=None, chunksize=1, **kwargs):
        if not parallelable('max_workers', self.max_workers, f): self.max_workers = 0
        self.not_parallel = self.max_workers==0
        if self.not_parallel: self.max_workers=1

        if self.not_parallel == False: self.lock = Manager().Lock()
        g = partial(f, *args, **kwargs)
        if self.not_parallel: return map(g, items)
        _g = partial(_call, self.lock, self.pause, self.max_workers, g)
        try: return super().map(_g, items, timeout=timeout, chunksize=chunksize)
        except Exception as e: self.on_exc(e)

In [ ]:
show_doc(ProcessPoolExecutor, title_level=4)

---

[source](https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/parallel.py#L95){target="_blank" style="float:right; font-size:smaller"}

#### ProcessPoolExecutor

```python

def ProcessPoolExecutor(
    max_workers:int=16, on_exc:builtin_function_or_method=print, pause:int=0, mp_context:NoneType=None,
    initializer:NoneType=None, initargs:tuple=(), max_tasks_per_child:NoneType=None
):


```

*Same as Python's ProcessPoolExecutor, except can pass `max_workers==0` for serial execution*

We define `import_progress_bar` to lazily import, `fastprogress` depends on `fasthtml` and takes about 1 sec to load

In [ ]:
#| export
def import_progress_bar():
    try: from fastprogress import progress_bar
    except ImportError: return None
    return progress_bar

In [ ]:
#| export
def parallel(f, items, *args, n_workers=defaults.cpus, total=None, progress=None, pause=0,
             method=None, threadpool=False, timeout=None, chunksize=1, **kwargs):
    "Applies `func` in parallel to `items`, using `n_workers`"
    progress_bar = import_progress_bar()
    kwpool = {}
    if threadpool: pool = ThreadPoolExecutor
    else:
        if not method and sys.platform == 'darwin': method='fork'
        if method: kwpool['mp_context'] = get_context(method)
        pool = ProcessPoolExecutor
    with pool(n_workers, pause=pause, **kwpool) as ex:
        r = ex.map(f,items, *args, timeout=timeout, chunksize=chunksize, **kwargs)
        if progress and progress_bar:
            if total is None: total = len(items)
            r = progress_bar(r, total=total, leave=False)
        return L(r)

In [ ]:
#| export
def _add_one(x, a=1):
    # this import is necessary for multiprocessing in notebook on windows
    import random
    time.sleep(random.random()/80)
    return x+a

In [ ]:
inp,exp = range(50),range(1,51)

test_eq(parallel(_add_one, inp, n_workers=2), exp)
test_eq(parallel(_add_one, inp, threadpool=True, n_workers=2), exp)
test_eq(parallel(_add_one, inp, n_workers=1, a=2), range(2,52))
test_eq(parallel(_add_one, inp, n_workers=0), exp)
test_eq(parallel(_add_one, inp, n_workers=0, a=2), range(2,52))

Use the `pause` parameter to ensure a pause of `pause` seconds between processes starting. This is in case there are race conditions in starting some process, or to stagger the time each process starts, for example when making many requests to a webserver. Set `threadpool=True` to use `ThreadPoolExecutor` instead of `ProcessPoolExecutor`.

In [ ]:
from datetime import datetime

In [ ]:
def print_time(i): 
    time.sleep(random.random()/1000)
    print(i, datetime.now())

parallel(print_time, range(5), n_workers=2, pause=0.1);

0

2026-05-02 06:18:54.278913

1

2026-05-02 06:18:54.380686

2

2026-05-02 06:18:54.481676

3

2026-05-02 06:18:54.584016

4

2026-05-02 06:18:54.684495

In [ ]:
#| hide
def die_sometimes(x):
#     if 3<x<6: raise Exception(f"exc: {x}")
    return x*2

parallel(die_sometimes, range(8))

[0, 2, 4, 6, 8, 10, 12, 14]

In [ ]:
#| export
async def parallel_async(f, items, *args, n_workers=16, pause=0,
                         timeout=None, chunksize=1, on_exc=print, cancel_on_error=False, **kwargs):
    "Applies `f` to `items` in parallel using asyncio and a semaphore to limit concurrency."
    semaphore = asyncio.Semaphore(n_workers)
    async def limited_task(i, item):
        coro = f(item, *args, **kwargs) if asyncio.iscoroutinefunction(f) else asyncio.to_thread(f, item, *args, **kwargs)
        if pause: await asyncio.sleep(i * pause)
        async with semaphore:
            return await asyncio.wait_for(coro, timeout) if timeout else await coro
    if cancel_on_error:
        async with asyncio.TaskGroup() as tg: tasks = [tg.create_task(limited_task(i, item)) for i,item in enumerate(items)]
        return [t.result() for t in tasks]
    return await asyncio.gather(*[limited_task(i, item) for i,item in enumerate(items)], return_exceptions=True)

In [ ]:
async def print_time_async(i): 
    start =datetime.now()
    wait = random.random()/30
    await asyncio.sleep(wait)
    print(i, start, datetime.now(), wait)

await parallel_async(print_time_async, range(6), n_workers=3);

0 2026-05-02 06:18:54.863372 2026-05-02 06:18:54.879976 0.015635678708191197
1 2026-05-02 06:18:54.863397 2026-05-02 06:18:54.882997 0.019249022668796727
2 2026-05-02 06:18:54.863403 2026-05-02 06:18:54.887730 0.023758669309642773
3 2026-05-02 06:18:54.880132 2026-05-02 06:18:54.900268 0.01907637860895647
4 2026-05-02 06:18:54.883061 2026-05-02 06:18:54.900419 0.01732184794923949
5 2026-05-02 06:18:54.887778 2026-05-02 06:18:54.903026 0.014917916227832732


Adding `pause` ensures a gap between starts:

In [ ]:
await parallel_async(print_time_async, range(6), n_workers=3, pause=0.1);

0 2026-05-02 06:18:54.926304 2026-05-02 06:18:54.943307 0.015931215772872948
1 2026-05-02 06:18:55.027303 2026-05-02 06:18:55.033705 0.005639884521666661


2

 2026-05-02 06:18:55.127385 2026-05-02 06:18:55.144286 0.01569178427437135
3 2026-05-02 06:18:55.227448 2026-05-02 06:18:55.257822 0.02922957194858668
4 2026-05-02 06:18:55.327466 2026-05-02 06:18:55.337080 0.008768163173784578


5 2026-05-02 06:18:55.427390 2026-05-02 06:18:55.456955 0.028371390226150824


In [ ]:
# With the default `cancel_on_error=False`, all tasks run and exceptions are returned

In [ ]:
async def maybe_fail(i:int):
    "Double i unless it's 3, in which case fail"
    await asyncio.sleep(random.random()/50)
    if i==3: raise ValueError(f"bad: {i}")
    return i*2

await parallel_async(maybe_fail, range(6), n_workers=3)

[0, 2, 4, ValueError('bad: 3'), 8, 10]

With `cancel_on_error=True`, `parallel_async` cancels remaining on first failure:

In [ ]:
try: res = await parallel_async(maybe_fail, range(6), n_workers=3, cancel_on_error=True)
except ExceptionGroup as e:
    print(f"Exception: {e}")
    print(f"Inner exceptions: {e.exceptions}")

Exception: unhandled errors in a TaskGroup (1 sub-exception)
Inner exceptions: (ValueError('bad: 3'),)


In [ ]:
#| export
def bg_task(coro):
    "Like `asyncio.create_task` but logs exceptions for fire-and-forget tasks"
    import traceback,asyncio
    def _done(t):
        if not t.cancelled() and (exc := t.exception()): traceback.print_exception(exc)
    task = asyncio.create_task(coro)
    task.add_done_callback(_done)
    return task

In [ ]:
async def _ok(): return 42
async def _fail(): raise ValueError("this error will be printed")

t1 = bg_task(_ok())
t2 = bg_task(_fail())
await asyncio.sleep(0.01)
test_eq(t1.result(), 42)

Traceback (most recent call last):
  File "/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipykernel_35266/1179310912.py", line 2, in _fail
    async def _fail(): raise ValueError("this error will be printed")
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: this error will be printed


In [ ]:
#| export
def run_procs(f, f_done, args):
    "Call `f` for each item in `args` in parallel, yielding `f_done`"
    Proc = get_context('fork').Process if sys.platform == 'darwin' else Process
    processes = L(args).map(Proc, args=arg0, target=f)
    for o in processes: o.start()
    yield from f_done()
    processes.map(Self.join())

In [ ]:
#| export
def _f_pg(obj, queue, batch, start_idx):
    for i,b in enumerate(obj(batch)): queue.put((start_idx+i,b))

def _done_pg(queue, items): return (queue.get() for _ in items)

In [ ]:
#| export
def parallel_gen(cls, items, n_workers=defaults.cpus, progress=True, **kwargs):
    "Instantiate `cls` in `n_workers` procs & call each on a subset of `items` in parallel."
    progress_bar = import_progress_bar()
    if not parallelable('n_workers', n_workers): n_workers = 0
    if n_workers==0:
        yield from enumerate(list(cls(**kwargs)(items)))
        return
    batches = L(chunked(items, n_chunks=n_workers))
    idx = L(itertools.accumulate(0 + batches.map(len)))
    queue = Queue()
    if progress_bar and progress: items = progress_bar(items, leave=False)
    f=partial(_f_pg, cls(**kwargs), queue)
    done=partial(_done_pg, queue, items)
    yield from run_procs(f, done, L(batches,idx).zip())

`cls` is any class with `__call__`. It will be passed `args` and `kwargs` when initialized. Note that `n_workers` instances of `cls` are created, one in each process. `items` are then split in `n_workers` batches and one is sent to each `cls`. The function then returns a generator of tuples of item indices and results.

In [ ]:
class TestSleepyBatchFunc:
    "For testing parallel processes that run at different speeds"
    def __init__(self): self.a=1
    def __call__(self, batch):
        for k in batch:
            time.sleep(random.random()/10)
            yield k+self.a

x = np.linspace(0,0.99,20)

res = L(parallel_gen(TestSleepyBatchFunc, x, n_workers=2, progress=False))
test_eq(res.sorted().itemgot(1), x+1)

In [ ]:
#| hide
# check that parallelism doesn't change the output
class _C:
    def __call__(self, o): return ((i+1) for i in o)

items = range(5)

res = L(parallel_gen(_C, items, n_workers=0, progress=False))
idxs,dat1 = zip(*res.sorted(itemgetter(0)))
test_eq(dat1, range(1,6))

res = L(parallel_gen(_C, items, n_workers=3, progress=False))
idxs,dat2 = zip(*res.sorted(itemgetter(0)))
test_eq(dat2, dat1)

In [ ]:
#|hide
# from subprocess import Popen, PIPE
# # test num_workers > 0 in scripts works when python process start method is spawn
# process = Popen(["python", "parallel_test.py"], stdout=PIPE)
# _, err = process.communicate(timeout=10)
# exit_code = process.wait()
# test_eq(exit_code, 0)

# Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()